# Zebrafish embryogenesis

This notebook shows the complete `zebrafish` analysis: data preparation,
model training, downstream analysis, and the commands used for the paper
figures. Edit the paths in **Setup** before starting a run.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)
DATASET_CONFIG = 'zebrafish'
RAW_H5AD = Path("data/zebrafish_raw.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/zebrafish")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / 'zebrafish_aligned.h5ad'
MODEL_DIR = OUTPUT_DIR / "training"


RUN_PREPARATION = False
RUN_PREPROCESS_AND_TRAIN = False
RUN_DOWNSTREAM = False

In [2]:
config, config_source = load_workflow_config(DATASET_CONFIG)
dataset = config["dataset"]
scientific = config["scientific"]
downstream = config["downstream"]

pd.DataFrame(
    {
        "setting": [
            "dataset",
            "configuration",
            "raw time column",
            "cell annotation",
            "observed training times",
            "classifier neighbors",
        ],
        "value": [
            dataset["display_name"],
            config_source,
            config["preprocess"]["time_key"],
            dataset["annotation_key"],
            ", ".join(map(str, downstream["observed"])),
            scientific["classifier_k"],
        ],
    }
)

,setting,value
0,dataset,Zebrafish embryogenesis
1,configuration,example configuration: zebrafish
2,raw time column,time
3,cell annotation,Annotation
4,observed training times,"0.0, 1.0, 2.0, 3.0, 4.0"
5,classifier neighbors,10


## Data preparation

The dataset configuration records the count layer, time mapping, spatial
coordinates, and alignment settings. The command below reads the raw H5AD and
writes the aligned H5AD and edge model used for training.

### 1. preprocess

```text
cytobridge workflow --config zebrafish --step preprocess --input-h5ad <raw.h5ad> --output-dir <run>
```

Input: `raw H5AD and the dataset configuration`

Creates: `<run>/preprocess/zebrafish_aligned.h5ad; <run>/preprocess/edge_classifier/zebrafish_edge_model.pt; preprocessing records`

Continue with: `training`

In [3]:
preparation_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess",),
)
preparation_plan = build_workflow_plan(
    config,
    source=config_source,
    options=preparation_options,
)
print(render_workflow_plan(preparation_plan))

CytoBridge workflow plan
dataset: Zebrafish embryogenesis (zebrafish)
config: example configuration: zebrafish
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/zebrafish/preprocess/zebrafish_aligned.h5ad
    edge predictor: not requested during preprocessing
  train: skipped; add --train to run (GPU required for training)
  downstream: skipped (GPU recommended)


In [4]:
if RUN_PREPARATION:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before preprocessing: {RAW_H5AD}")
    preparation_result = run_workflow(config, options=preparation_options)
    preparation_result
else:
    print("Data preparation is off. Set RUN_PREPARATION = True to run it.")

Data preparation is off. Set RUN_PREPARATION = True to run it.


## Training

The full run starts from the raw H5AD, writes the aligned data, fits the
interaction edge model when needed, and trains CytoBridge. Training requires a
CUDA-capable environment.

### 1. preprocess and train

```text
cytobridge workflow --config zebrafish --step preprocess --step train --train --input-h5ad <raw.h5ad> --output-dir <run> --device cuda
```

Input: `raw H5AD, dataset configuration, and LR database`

Creates: `<run>/training/<stage>/best_model.pth or score_model.pth; <run>/training/adata.h5ad; training_history.csv; training_run_summary.json`

Continue with: `downstream`

In [5]:
training_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess", "train"),
    train=True,
)
training_plan = build_workflow_plan(
    config,
    source=config_source,
    options=training_options,
)
print(render_workflow_plan(training_plan))

CytoBridge workflow plan
dataset: Zebrafish embryogenesis (zebrafish)
config: example configuration: zebrafish
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/zebrafish/preprocess/zebrafish_aligned.h5ad
    edge predictor: will be trained automatically
      graph database: package: CytoBridge/workflow_databases/CellChatDB.ligrec.zebrafish.csv
      database source: included CellChatDB resource
      interaction cutoff: 0.09606367405591873
      decision threshold source: validation-selected during de novo training
      output: tutorial_outputs/zebrafish/preprocess/edge_classifier/zebrafish_edge_model.pt
  train: ready (GPU required for training)
    training config: zebrafish_spatial_full_alpha_express_0015.yaml
    interaction cutoff: 0.09606367405591873
    edge predictor threshold source: validation-selected during preprocessing
    edge predictor: tutori

In [6]:
if RUN_PREPROCESS_AND_TRAIN:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before training: {RAW_H5AD}")
    training_result = run_workflow(config, options=training_options)
    training_result
else:
    print("Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.")

Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.


## Downstream analysis

Downstream analysis reads the aligned H5AD and fitted model from the training
directory. It writes generated states, velocity, growth, composition,
communication, ligand–receptor tables, and standard figures.

### 1. downstream

```text
cytobridge workflow --config zebrafish --step downstream --aligned-h5ad <run>/preprocess/zebrafish_aligned.h5ad --model-dir <run>/training --output-dir <run>
```

Input: `aligned H5AD; <run>/training; dataset-matched LR database`

Creates: `<run>/downstream/summary.json; slice_data/*.h5ad; velocity/velocity_components.npz; growth/growth_by_cell.csv; composition/celltype_composition.csv; communication and ligand_receptor tables; standard figures`

Continue with: `paper-specific continuation shown in the paper-figure notebook`

In [7]:
downstream_options = WorkflowOptions(
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    output_dir=OUTPUT_DIR,
    steps=("downstream",),
)
downstream_plan = build_workflow_plan(
    config,
    source=config_source,
    options=downstream_options,
)
print(render_workflow_plan(downstream_plan))

CytoBridge workflow plan
dataset: Zebrafish embryogenesis (zebrafish)
config: example configuration: zebrafish
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: skipped (GPU for spatial alignment)
  train: skipped; add --train to run (GPU required for training)
  downstream: ready (GPU recommended for SDE simulation and classifier fitting)
    model format: current
    output: tutorial_outputs/zebrafish/downstream
    generated states: observed times=[0.0, 1.0, 2.0, 3.0, 4.0], additional times=[0.5, 1.5, 2.5, 3.5]
      simulation settings: dt=0.05, sigma=0.03, daughter noise=0, growth alpha=1
      each additional time starts from the preceding observed time point
    interpolation and classification: enabled
    time-slice velocity: enabled
    growth: enabled when present in the model
    cell-type composition: enabled
    sparse communication: enabled
    standard figures: enabled
      note: snapshots, mosaic, growth, composition,

In [8]:
if RUN_DOWNSTREAM:
    missing = [path for path in (ALIGNED_H5AD, MODEL_DIR) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing aligned data or model directory: {missing}")
    downstream_result = run_workflow(config, options=downstream_options)
    downstream_result
else:
    print("Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.")

Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.


## Paper figures

Continue with these commands to calculate the values used in the paper. Each
step states which downstream files it reads and which paper notebook uses its
output.

- [Supplementary Figures S31–S38](../paper_figures/zebrafish_si_s31_s38.ipynb)
- [Supplementary Figure S43](../paper_figures/zebrafish_attention.ipynb)

### 1. calculate the zebrafish analyses

Used for: S31-S35; S38

```text
python -m scripts.run_zebrafish_paper_downstream --aligned-h5ad <run>/preprocess/zebrafish_aligned.h5ad --model-dir <run>/training --acceptance-report <run>/matched_ablation_acceptance.json --lr-database <zebrafish-lr.csv> --output-dir <paper-run> --stage all --device cuda
```

Input: `aligned H5AD, six-stage model, validation JSON, and zebrafish LR database`

Creates: `global-t0 state transport, growth, virtual-removal, gene-dynamics, inverse-PCA, and communication tables`

Continue with: `use the S31-S38 notebook to calculate panel values and draw the figures`

Use the validation value written beside the trained model so the data and model come from the same run.

### 2. run loss-weight sensitivity

Used for: S36

```text
python scripts/paper_figures/zebrafish_loss_weight/prepare_configs.py --base-config <zebrafish-training.yaml> --output-dir <loss-config-dir>
```

Input: `base training YAML`

Creates: `one training YAML per loss setting`

Continue with: `train each YAML with the command shown in the S31-S38 notebook`

### 3. run daughter-noise sensitivity

Used for: S37

```text
python -m scripts.run_zebrafish_interval_daughter_noise_sensitivity --aligned-h5ad <run>/preprocess/zebrafish_aligned.h5ad --model-dir <run>/training --classifier-cache <paper-run>/classifier/classifier.pt --acceptance-report <run>/matched_ablation_acceptance.json --output-dir <daughter-noise-run> --device cuda:0
```

Input: `zebrafish model and fixed evaluation cells`

Creates: `daughter-noise composition, lineage, and particle-count tables`

Continue with: `draw S37 and run the S31-S38 paper notebook`

Draw S37 from the completed analysis:

```text
python scripts/plot_zebrafish_interval_daughter_noise_sensitivity.py --run-manifest <daughter-noise-run>/run_manifest.json --acceptance-report <run>/matched_ablation_acceptance.json --output-dir <daughter-noise-figure>
```

### 4. calculate attention validation

Used for: S43

```text
python -m scripts.run_zebrafish_attention_validation analyze --spec <analysis-spec.json> --output-dir <attention-analysis> --n-selected-pairs 30
```

Input: `model attention, aligned cells, LR pairs, COMMOT results, and CellAgentChat results`

Creates: `directed-pair, expression, display-edge, and interaction-sensitivity tables`

Continue with: `run the report command, then use the S43 notebook`

## Saved files

- Aligned data: `tutorial_outputs/zebrafish/preprocess/zebrafish_aligned.h5ad`
- Training directory: `tutorial_outputs/zebrafish/training`
- Downstream directory: `tutorial_outputs/zebrafish/downstream`